# Generate survey_questions.json for the Staff Survey API
Reads `dbo.survey_questions` from the Lakehouse, corrects Q18 branching logic, and writes a JSON file to `Files/Question_data_json/survey_questions.json`.

In [ ]:
TABLE_PATH  = "abfss://RCD_StaffSurvey@onelake.dfs.fabric.microsoft.com/Staff_survey_HDFT.Lakehouse/Tables/dbo/survey_questions"
OUTPUT_PATH = "abfss://RCD_StaffSurvey@onelake.dfs.fabric.microsoft.com/Staff_survey_HDFT.Lakehouse/Files/Question_data_json/survey_questions.json"
SURVEY_TITLE = "HDFT Staff Survey 2026"

In [ ]:
from pyspark.sql import functions as F

df = spark.read.format("delta").load(TABLE_PATH)
print(f"Read {df.count()} rows")
df.printSchema()
df.orderBy("sequence").show(5, truncate=False)

In [ ]:
# Q18 sub-questions were always_show with no show_if_answer — should only appear when Q18 = 'Yes'
df_fixed = (
    df
    .withColumn(
        "branch_type",
        F.when(
            (F.col("parent_question_id") == "Q18") & (F.col("branch_type") == "always_show"),
            F.lit("conditional")
        ).otherwise(F.col("branch_type"))
    )
    .withColumn(
        "show_if_answer",
        F.when(
            (F.col("parent_question_id") == "Q18") & F.col("show_if_answer").isNull(),
            F.lit("Yes")
        ).otherwise(F.col("show_if_answer"))
    )
)

print("Q18 sub-questions after fix:")
df_fixed.filter(F.col("parent_question_id") == "Q18") \
    .select("question_id", "question_text", "branch_type", "show_if_answer") \
    .show(truncate=False)

In [ ]:
import json
from datetime import datetime

rows = df_fixed.orderBy("sequence").collect()
questions = []

for row in rows:
    q = {}
    for field in row.__fields__:
        val = row[field]
        if field == "answer_options" and isinstance(val, str):
            try:
                val = json.loads(val)
            except Exception:
                val = []
        if field == "is_required":
            val = bool(val)
        if hasattr(val, "isoformat"):
            val = val.isoformat()
        q[field] = val
    questions.append(q)

survey_data = {
    "survey_title": SURVEY_TITLE,
    "generated_at": datetime.utcnow().isoformat(),
    "total_questions": len(questions),
    "questions": questions
}

print(f"Built JSON with {len(questions)} questions")
for q in questions[:5]:
    print(f"  [{q['sequence']}] {q['question_id']}: branch_type={q.get('branch_type')}, show_if_answer={q.get('show_if_answer')}")

In [ ]:
json_str = json.dumps(survey_data, indent=2, default=str)
notebookutils.fs.put(OUTPUT_PATH, json_str, overwrite=True)
print(f"Written {len(json_str):,} characters to:\n{OUTPUT_PATH}")

In [ ]:
preview = notebookutils.fs.head(OUTPUT_PATH, 1000)
print(preview)